## <center> دورة التعلم الآلي المفتوحة mlcourse.ai. جلسة اللغة الانجليزية رقم 1
### <center> المؤلف: فالنتين كوفاليف



## <center> البرنامج التعليمي </center>
### <center> تخصيص ديريشليت الكامن </center>



في هذا البرنامج التعليمي سأحاول بعض تخصيص Dirichlet الخفي لاستخراج المواضيع التي تميز النصوص تلقائيًا. <br> الضبط الجيد لـ LDA (هذا فن) يمكن أن يعطي نتيجة جيدة حقًا على لوحة المتصدرين في مسابقات kaggle ذات الميزات النصية. <br>



#### استيراد المكتبات


In [ ]:
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.test.utils import common_texts


### استخراج المواضيع باستخدام LDA
يمثل LDA المستندات كمزيج من الموضوعات التي تلفظ كلمات ذات احتمالات معينة. <br>
بالنسبة لكل موضوع محتمل Z، سنضرب تكرار هذه الكلمة من النوع W في Z بعدد الكلمات الأخرى في المستند D التي تنتمي بالفعل إلى Z. وستمثل النتيجة احتمال أن هذه الكلمة جاءت من Z.



### تدريب نموذج LDA باستخدام مجموعة Gensim



#### إنشاء مجموعة من قائمة النصوص


In [ ]:
common_dictionary = Dictionary(common_texts)
common_corpus = [common_dictionary.doc2bow(text) for text in common_texts]


#### تدريب النموذج على الجسم.


In [ ]:
lda = LdaModel(common_corpus, num_topics=10)


### يمكننا حفظ نموذج على القرص، أو إعادة تحميل نموذج تم تدريبه مسبقًا
سيتم التعليق على هذا الرمز لعدم إنتاج الكيانات


In [ ]:
from gensim.test.utils import datapath


#### حفظ النموذج على القرص


In [ ]:
# temp_file = datapath("model")
# lda.save(temp_file)


#### قم بتحميل نموذج تم تدريبه مسبقًا من القرص.


In [ ]:
# lda = LdaModel.load(temp_file)


### التحقق من النموذج عند استخدام المستندات الجديدة غير المرئية



#### قم بإنشاء مجموعة جديدة مكونة من مستندات غير مرئية من قبل.


In [ ]:
other_texts = [
    ["computer", "time", "graph"],
    ["survey", "response", "eps"],
    ["human", "system", "computer"],
]
other_corpus = [common_dictionary.doc2bow(text) for text in other_texts]

unseen_doc = other_corpus[0]
vector = lda[unseen_doc]  # get topic probability distribution for a document


### قم بتحديث النموذج من خلال التدريب المتزايد على المجموعة الجديدة


In [ ]:
lda.update(other_corpus)
vector = lda[unseen_doc]


#### حول المعلمات الفائقة


معلمات ألفا وبيتا الفائقة - تمثل ألفا كثافة موضوع المستند وتمثل بيتا كثافة كلمات الموضوع. كلما زادت قيمة ألفا، تتكون المستندات من موضوعات أكثر، وكلما انخفضت قيمة ألفا، تحتوي المستندات على موضوعات أقل. من ناحية أخرى، كلما ارتفعت قيمة بيتا، تتكون المواضيع من عدد كبير من الكلمات في المجموعة، ومع انخفاض قيمة بيتا، فإنها تتكون من كلمات قليلة.
عدد المواضيع - عدد المواضيع التي سيتم استخراجها من المجموعة. لقد طور الباحثون أساليب للحصول على العدد الأمثل من المواضيع باستخدام نقاط تباعد كولباك ليبلر. لن أناقش هذا بالتفصيل، لأنه رياضي للغاية. للفهم، يمكن للمرء الرجوع إلى هذه الورقة الأصلية [1] حول استخدام تباعد KL.
عدد مصطلحات الموضوع - عدد المصطلحات المكونة في موضوع واحد. يتم تحديده بشكل عام وفقًا للمتطلبات. إذا كان بيان المشكلة يتحدث عن استخلاص السمات أو المفاهيم، فمن المستحسن اختيار رقم أعلى، إذا كان بيان المشكلة يتحدث عن استخلاص الميزات أو المصطلحات، فمن المستحسن اختيار رقم منخفض.
عدد التكرارات/التمريرات - الحد الأقصى لعدد التكرارات المسموح بها لخوارزمية LDA للتقارب.



### حسنًا، لنتحقق من البيانات الحقيقية


In [ ]:
doc1 = "Sugar is bad to consume. My sister likes to have sugar, but not my father."
doc2 = "My father spends a lot of time driving my sister around to dance practice."
doc3 = "Doctors suggest that driving may cause increased stress and blood pressure."
doc4 = "Sometimes I feel pressure to perform well at school, but my father never seems to drive my sister to do better."
doc5 = "Health experts say that Sugar is not good for your lifestyle."

# compile documents
doc_complete = [doc1, doc2, doc3, doc4, doc5]


#### التنظيف والمعالجة المسبقة


In [ ]:
import string

from nltk.corpus import stopwords
from nltk.stem.wordnet import WordNetLemmatizer

stop = set(stopwords.words("english"))
exclude = set(string.punctuation)
lemma = WordNetLemmatizer()


def clean(doc):
    stop_free = " ".join([i for i in doc.lower().split() if i not in stop])
    punc_free = "".join(ch for ch in stop_free if ch not in exclude)
    normalized = " ".join(lemma.lemmatize(word) for word in punc_free.split())
    return normalized


doc_clean = [clean(doc).split() for doc in doc_complete]


#### إعداد مصفوفة مدة الوثيقة


In [ ]:
# Importing Gensim
import gensim
from gensim import corpora

# Creating the term dictionary of our courpus, where every unique term is assigned an index.
dictionary = corpora.Dictionary(doc_clean)

# Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]


#### تشغيل نموذج LDA


In [ ]:
# Creating the object for LDA model using gensim library
Lda = gensim.models.ldamodel.LdaModel

# Running and Trainign LDA model on the document term matrix.
ldamodel = Lda(doc_term_matrix, num_topics=3, id2word=dictionary, passes=50)


#### النتائج


In [ ]:
print(ldamodel.print_topics(num_topics=3, num_words=3))


### نمذجة الموضوع لاختيار الميزة


في بعض الأحيان يمكن أيضًا استخدام LDA كتقنية لاختيار الميزات. خذ مثالاً على مشكلة تصنيف النص حيث تحتوي بيانات التدريب على مستندات فئة حكيمة. إذا كان LDA يعمل على مجموعات من المستندات الحكيمة للفئة. متبوعًا بإزالة مصطلحات الموضوع الشائعة عبر نتائج الفئات المختلفة، ستوفر أفضل الميزات للفئة.



### المكافأة: pyLDAvis



تم تصميم pyLDAvis لمساعدة المستخدمين على تفسير المواضيع في نموذج موضوع ملائم لمجموعة البيانات النصية. تستخرج الحزمة المعلومات من نموذج موضوع LDA المجهز لإبلاغ التصور التفاعلي على شبكة الإنترنت.
تم تصميم التصور ليتم استخدامه داخل دفتر IPython ولكن يمكن أيضًا حفظه في ملف HTML مستقل لتسهيل المشاركة.



افتراضيًا، يتم عرض المواضيع على المستوى ثنائي الأبعاد باستخدام PCoA على مصفوفة المسافة التي تم إنشاؤها باستخدام تباعد Jensen-Shannon على توزيعات مصطلح الموضوع. يمكنك تمرير وظيفة تحجيم مختلفة متعددة الأبعاد عبر معلمة mds. بالإضافة إلى PCoa، هناك خيارات أخرى متوفرة وهي tsne وmmds التي تعمل على نفس مصفوفة مسافة التباعد JS. يتطلب كل من tsne وmmds تثبيت sklearn. هنا tnse في العمل:


In [ ]:
import pyLDAvis.gensim

vis = pyLDAvis.gensim.prepare(ldamodel, corpus=doc_term_matrix, dictionary=dictionary)
pyLDAvis.display(vis)


### المراجع:
- <a href="@@KEEP_00000@@> موقع LDA الرسمي</a>.
- <a href="@@KEEP_00001@@> LDA مع Python</a>.
- <a href="@@KEEP_00002@@> LDA على دليل بايثون</a>.
- <a href="@@KEEP_00003@@>اختبار على نماذج جينسيم</a>.
- <a href="@@KEEP_00004@@> مكتبة pyLDAvis</a>.